# Bonds

Author: Sebastien Gurrieri, sebgur@gmail.com

Bond definitions, with calculation of typical quantities (e.g. yields, spreads, durations). 

In [1]:
import datetime as dt
import pandas as pd
import numpy as np
# import matplotlib.pyplot as plt
import sdevpy as sd
from sdevpy import pricingcontext
# from sdevpy.utilities import dates as dts
from sdevpy.utilities.scalendar import BDC, make_calendar
from sdevpy.utilities.sdaycount import DayCount
from sdevpy.utilities import schedules as sch
# from sdevpy.utilities import timegrids
from sdevpy.rates import bonds, swaps
from sdevpy.calibration.rates import yieldcurve as ycrv


print("SDevPy version: " + sd.__version__)
ctx = pricingcontext.default_pricing_context()

SDevPy version: 1.2


### Static Definition
Definition of the bond from static inputs (dates, conventions, etc.). Display of the cash-flow schedule and accruals.

In [7]:
# Static inputs
issue = dt.date(2025, 1, 10)
maturity = dt.date(2036, 2, 15)
coupon = 0.04
conv = sch.LegConvention(frequency='6M', daycount=DayCount.ACTACT_ICMA, calendar='USD', bdc=BDC.F,
                     adjust_accrual=False, eom=True)

# Schedule
scheduler = sch.make_bond_scheduler(issue, maturity, conv)
cpn_schedule = scheduler.coupon_schedule(coupon)
schedule_df = pd.DataFrame(cpn_schedule)

print(f"Bond {coupon:.2%} {maturity:%d-%b-%Y}: {scheduler.schedule.n_periods} coupons,"
      f" {scheduler.coupons_per_year} per year, redemption on {scheduler.maturity}")
date_fmt = '{:%d-%b-%Y}'
schedule_df.style.format({'Accrual start': date_fmt, 'Accrual end': date_fmt, 'Payment': date_fmt,
                       'Accrual fraction': '{:.6f}', 'Coupon': '{:.6f}'})


Bond 4.00% 15-Feb-2036: 23 coupons, 2 per year, redemption on 2036-02-15


,Accrual start,Accrual end,Payment,Days,Accrual fraction,Coupon
0,10-Jan-2025,15-Feb-2025,18-Feb-2025,36,0.097826,0.391304
1,15-Feb-2025,15-Aug-2025,15-Aug-2025,181,0.500000,2.000000
2,15-Aug-2025,15-Feb-2026,17-Feb-2026,184,0.500000,2.000000
3,15-Feb-2026,15-Aug-2026,17-Aug-2026,181,0.500000,2.000000
4,15-Aug-2026,15-Feb-2027,16-Feb-2027,184,0.500000,2.000000
5,15-Feb-2027,15-Aug-2027,16-Aug-2027,181,0.500000,2.000000
6,15-Aug-2027,15-Feb-2028,15-Feb-2028,184,0.500000,2.000000
7,15-Feb-2028,15-Aug-2028,15-Aug-2028,182,0.500000,2.000000
8,15-Aug-2028,15-Feb-2029,15-Feb-2029,184,0.500000,2.000000
9,15-Feb-2029,15-Aug-2029,15-Aug-2029,181,0.500000,2.000000


### Price-Based Calculations
Now we assume the bond price is given on a certain valuation date and we calculate a number of relevant quantities for the bond.

In [9]:
# Market inputs
valdate = dt.datetime(2025, 12, 15)
clean_price = 98.50
settlement = make_calendar(conv.calendar).add_business_days(valdate.date(), 1)  # UST: T+1

# Retrieve yield curve
dataset = ctx.calib_repo[valdate]
curve = dataset.get_yieldcurve("USD.SOFR.1D")
df = curve.discount_float

# Price, accrued, yield
bf = bonds.bond_flows(scheduler, settlement, valdate)
accrued = bonds.accrued_interest(bf, coupon)
dirty = clean_price + accrued
ytm = bonds.yield_from_dirty(bf, coupon, dirty)
starts, ends = scheduler.accrual_dates()
i = int(np.searchsorted(ends, np.datetime64(settlement, 'D'), side='right'))

# Spreads: Z-spread over the curve, I-spread over the par swap rate to the bond's maturity
zspread = bonds.z_spread(bf, coupon, dirty, df)
sofr = sch.LegConvention('1Y', DayCount.ACT360, 'USD', pay_lag=2)
usd_ois = sch.SwapConvention(sofr, sofr)
spot = make_calendar('USD').add_business_days(valdate.date(), usd_ois.spot_lag)
swap_rate = swaps.par_rate(swaps.swap_times(sch.make_swap_scheduler(spot, maturity, usd_ois), valdate), df)

analytics = {
    'Settlement': f"{settlement:%d-%b-%Y}",
    'Previous coupon': f"{sch.to_date(starts[i]):%d-%b-%Y}",
    'Next coupon': f"{sch.to_date(ends[i]):%d-%b-%Y}",
    'Days accrued': int((np.datetime64(settlement, 'D') - starts[i]).astype(int)),
    'Accrued interest': f"{accrued:.6f}",
    'Clean price': f"{clean_price:.6f}",
    'Dirty price': f"{dirty:.6f}",
    'Yield to maturity (street, s.a.)': f"{ytm:.4%}",
    'Yield (annual equivalent)': f"{(1 + ytm / bf.frequency) ** bf.frequency - 1:.4%}",
    'Current yield': f"{100 * coupon / clean_price:.4%}",
    'Macaulay duration': f"{bonds.macaulay_duration(bf, coupon, ytm):.4f}",
    'Modified duration': f"{bonds.modified_duration(bf, coupon, ytm):.4f}",
    'DV01 (per 100)': f"{bonds.dv01(bf, coupon, ytm):.6f}",
    'Convexity': f"{bonds.convexity(bf, coupon, ytm):.4f}",
    'Z-spread (bp)': f"{1e4 * zspread:.2f}",
    'Par swap rate to maturity': f"{swap_rate:.4%}",
    'I-spread (bp)': f"{1e4 * (ytm - swap_rate):.2f}",
}
pd.Series(analytics, name='Value').to_frame()

,Value
Settlement,16-Dec-2025
Previous coupon,15-Aug-2025
Next coupon,15-Feb-2026
Days accrued,123
Accrued interest,1.336957
Clean price,98.500000
Dirty price,99.836957
"Yield to maturity (street, s.a.)",4.1821%
Yield (annual equivalent),4.2258%
Current yield,4.0609%


### Curve-Based Calculations